# 6.1 Cross-Model R_pref Scoring (Databricks Companion)

Scores the 250 stratified AITA-YTA samples across 11 production LLM responses
using 8 reward models. Produces `data/cross_model_rpref.parquet` for merging
into the main cross-model analysis (Step 6).

**Input**: `data/cross_model_sampled.parquet` (from `6_cross_model_analysis.py` Stage 1)
**Output**: `data/cross_model_rpref.parquet` (auto-detected by `6_cross_model_analysis.py` Stage 3)

In [ ]:
# Cell 1: Imports, config, CUDA setup
import os
import sys
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig
from pathlib import Path
import numpy as np
from tqdm import tqdm
import warnings
import yaml
warnings.filterwarnings('ignore')

# Load configuration
config_path = Path('config.yaml')
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

REWARD_MODELS = config['step_3']['reward_models']
SCORING_CONFIG = config['step_3']['scoring']
STEP6_CONFIG = config['step_6']
MODEL_COLUMNS = STEP6_CONFIG['model_columns']

MAX_LENGTH = SCORING_CONFIG['max_length']
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device: {DEVICE}")
print(f"Reward models: {list(REWARD_MODELS.keys())}")
print(f"Production models to score: {MODEL_COLUMNS}")

# Load the sampled data from Stage 1
INPUT_PATH = "./data/cross_model_sampled.parquet"
OUTPUT_PATH = "./data/cross_model_rpref.parquet"

df = pd.read_parquet(INPUT_PATH)
print(f"\nLoaded {len(df)} sampled prompts")
print(f"Columns: {list(df.columns)}")

In [ ]:
# Cell 2: Download all reward models
from huggingface_hub import snapshot_download

def download_model(model_id: str, cache_dir: Path = None, revision: str = None, local=False) -> str:
    """
    Download a model from HuggingFace with custom endpoint configuration.
    """
    if cache_dir is None:
        cache_dir = Path("/local_disk0")
    else:
        cache_dir = Path(cache_dir)

    cache_dir.mkdir(parents=True, exist_ok=True)

    print(f"Downloading model: {model_id}")
    print(f"Cache directory: {cache_dir}")

    HF_ENDPOINT = "https://graingerreadonly.jfrog.io/artifactory/api/huggingfaceml/huggingfaceml-remote"

    try:
        args = {
            "repo_id": model_id,
            "etag_timeout": 86400,
            "force_download": False,
            "endpoint": HF_ENDPOINT,
        }
        if local:
            args["local_dir"] = str(cache_dir)
        else:
            args["cache_dir"] = str(cache_dir)

        if revision:
            args["revision"] = revision
        model_path = snapshot_download(**args)

        if not os.path.exists(model_path):
            raise FileNotFoundError(f"No files found at downloaded model path: {model_path}")
        if local:
            open(f"{model_path}/__init__.py", "a").close()
        print(f"Model downloaded successfully to: {model_path}")
        return model_path

    except Exception as e:
        print(f"Error downloading model {model_id}: {e}")
        raise


for i in REWARD_MODELS:
    src_model = REWARD_MODELS[i]['model_name']
    src_tokenizer = REWARD_MODELS[i]['tokenizer_name']

    dst_model = f"{SCORING_CONFIG['cache_dir']}/{i}"
    dst_tokenizer = f"{SCORING_CONFIG['cache_dir']}/{i}_tokenizer"

    try:
        download_model(src_model, cache_dir=dst_model, local=True)
        if src_model == src_tokenizer:
            continue
        download_model(src_tokenizer, cache_dir=dst_tokenizer, local=True)
    except Exception as e:
        print(f"   Error downloading {i}: {e}")
        continue

In [ ]:
# Cell 3: Register GPTNeoXRewardModel (needed for oasst_pythia_1b)
from transformers.models.gpt_neox.modeling_gpt_neox import (
    GPTNeoXConfig, GPTNeoXModel, GPTNeoXPreTrainedModel
)
import torch.nn as nn
from dataclasses import dataclass
from transformers.utils import ModelOutput


class GPTNeoXRewardModelConfig(GPTNeoXConfig):
    model_type = "gpt_neox_reward_model"


@dataclass
class GPTNeoXRewardModelOutput(ModelOutput):
    logits: torch.FloatTensor = None


class GPTNeoXRewardModel(GPTNeoXPreTrainedModel):
    config_class = GPTNeoXRewardModelConfig

    def __init__(self, config):
        if type(config) == GPTNeoXConfig:
            config = GPTNeoXRewardModelConfig.from_dict(config.to_dict())
        super().__init__(config)
        self.gpt_neox = GPTNeoXModel(config)
        self.out_proj = nn.Linear(config.hidden_size, 1)

    def forward(self, input_ids, attention_mask=None, **kwargs):
        outputs = self.gpt_neox(input_ids, attention_mask=attention_mask, **kwargs)
        hidden_states = outputs[0]
        pooled = hidden_states[:, -1]
        logits = self.out_proj(pooled)
        return GPTNeoXRewardModelOutput(logits=logits)


AutoConfig.register("gpt_neox_reward_model", GPTNeoXRewardModelConfig)
AutoModelForSequenceClassification.register(GPTNeoXRewardModelConfig, GPTNeoXRewardModel)
print("GPTNeoXRewardModel registered")

In [ ]:
# Cell 4: Scoring functions + main scoring loop
from transformers import pipeline


def format_prompt_response(prompt, response, tokenizer):
    """Format prompt and response using PersonalLLM's convention."""
    if tokenizer.chat_template is not None:
        try:
            formatted = tokenizer.apply_chat_template(
                [
                    {"role": "user", "content": prompt},
                    {"role": "assistant", "content": response},
                ],
                tokenize=False,
                add_generation_prompt=False,
            )
            if tokenizer.bos_token:
                formatted = formatted.replace(tokenizer.bos_token, "")
            return formatted
        except Exception:
            pass
    return f"User: {prompt}\nAssistant: {response}"


def score_responses_for_model(reward_model_name, prompts, responses):
    """Score a list of (prompt, response) pairs with one reward model."""
    print(f"   Loading {reward_model_name}...")

    try:
        model_config = REWARD_MODELS[reward_model_name]
        model_path = f"{SCORING_CONFIG['cache_dir']}/{reward_model_name}"
        if model_config['model_name'] == model_config['tokenizer_name']:
            tokenizer_path = model_path
        else:
            tokenizer_path = f"{SCORING_CONFIG['cache_dir']}/{reward_model_name}_tokenizer"

        tokenizer = AutoTokenizer.from_pretrained(tokenizer_path, trust_remote_code=True)
        model = AutoModelForSequenceClassification.from_pretrained(
            model_path,
            torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
        )
        model.eval()

        if tokenizer.pad_token_id is None:
            tokenizer.pad_token = tokenizer.eos_token
            tokenizer.pad_token_id = tokenizer.eos_token_id
        if model.config.pad_token_id is None:
            model.config.pad_token_id = tokenizer.pad_token_id

        reward_pipe = pipeline(
            "text-classification",
            model=model,
            tokenizer=tokenizer,
            device=DEVICE
        )

        batch_size = 32
        pipe_kwargs = {
            "batch_size": batch_size,
            "truncation": True,
            "padding": "longest",
            "max_length": MAX_LENGTH,
            "function_to_apply": "none",
            "return_token_type_ids": False,
        }

        formatted = [format_prompt_response(p, r, tokenizer) for p, r in zip(prompts, responses)]
        scores = []
        for i in tqdm(range(0, len(formatted), batch_size), desc=f"     {reward_model_name}"):
            batch_texts = formatted[i:i+batch_size]
            outputs = reward_pipe(batch_texts, **pipe_kwargs)

            if isinstance(outputs[0], dict):
                batch_scores = [o["score"] for o in outputs]
            elif isinstance(outputs[0][0], dict):
                batch_scores = [o[0]["score"] for o in outputs]
            else:
                batch_scores = [float(o[0].cpu().numpy()) if hasattr(o[0], 'cpu') else float(o[0]) for o in outputs]
            scores.extend(batch_scores)

        del model, tokenizer, reward_pipe
        torch.cuda.empty_cache()

        print(f"     mean={np.mean(scores):.3f}, std={np.std(scores):.3f}")
        return scores

    except Exception as e:
        import traceback
        print(f"     Error: {e}")
        print(traceback.format_exc())
        return [0.0] * len(prompts)


# Main scoring loop: 11 production models x 8 reward models
prompt_col = STEP6_CONFIG['prompt_col']
prompts = df[prompt_col].tolist()

# Collect all raw scores: key = (production_model, reward_model)
all_raw_scores = {}

for prod_model in MODEL_COLUMNS:
    if prod_model not in df.columns:
        print(f"WARNING: Production model column '{prod_model}' not found, skipping")
        continue

    responses = df[prod_model].fillna("").tolist()
    print(f"\n{'='*60}")
    print(f"Production model: {prod_model}")
    print(f"{'='*60}")

    for reward_model_name in REWARD_MODELS:
        scores = score_responses_for_model(reward_model_name, prompts, responses)
        all_raw_scores[(prod_model, reward_model_name)] = scores

print(f"\nScoring complete: {len(all_raw_scores)} (production, reward) combinations")

In [ ]:
# Cell 4.5: Retry failed (production_model, reward_model) pairs with smaller batch size
#
# Detects pairs that returned all-zeros (the fallback from OOM errors) and
# re-scores them one at a time with batch_size=4 and aggressive CUDA cleanup.

RETRY_BATCH_SIZE = 4

# Identify failed pairs: all_raw_scores entries where every value is 0.0
failed_pairs = []
for (pm, rm), scores in all_raw_scores.items():
    if all(s == 0.0 for s in scores):
        failed_pairs.append((pm, rm))

if not failed_pairs:
    print("No failed pairs detected — nothing to retry.")
else:
    print(f"Detected {len(failed_pairs)} failed (production_model, reward_model) pairs:")
    for pm, rm in failed_pairs:
        print(f"  {pm} + {rm}")

    prompt_col = STEP6_CONFIG['prompt_col']
    prompts = df[prompt_col].tolist()

    for pm, rm in failed_pairs:
        print(f"\n{'='*60}")
        print(f"RETRY: {pm} + {rm}  (batch_size={RETRY_BATCH_SIZE})")
        print(f"{'='*60}")

        # Aggressive cleanup before loading
        torch.cuda.empty_cache()
        import gc; gc.collect()

        responses = df[pm].fillna("").tolist()

        try:
            model_config = REWARD_MODELS[rm]
            model_path = f"{SCORING_CONFIG['cache_dir']}/{rm}"
            if model_config['model_name'] == model_config['tokenizer_name']:
                tokenizer_path = model_path
            else:
                tokenizer_path = f"{SCORING_CONFIG['cache_dir']}/{rm}_tokenizer"

            tokenizer = AutoTokenizer.from_pretrained(tokenizer_path, trust_remote_code=True)
            model = AutoModelForSequenceClassification.from_pretrained(
                model_path,
                torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
            )
            model.eval()

            if tokenizer.pad_token_id is None:
                tokenizer.pad_token = tokenizer.eos_token
                tokenizer.pad_token_id = tokenizer.eos_token_id
            if model.config.pad_token_id is None:
                model.config.pad_token_id = tokenizer.pad_token_id

            reward_pipe = pipeline(
                "text-classification",
                model=model,
                tokenizer=tokenizer,
                device=DEVICE
            )

            pipe_kwargs = {
                "batch_size": RETRY_BATCH_SIZE,
                "truncation": True,
                "padding": "longest",
                "max_length": MAX_LENGTH,
                "function_to_apply": "none",
                "return_token_type_ids": False,
            }

            formatted = [format_prompt_response(p, r, tokenizer) for p, r in zip(prompts, responses)]
            scores = []
            for i in tqdm(range(0, len(formatted), RETRY_BATCH_SIZE), desc=f"     {rm}"):
                batch_texts = formatted[i:i+RETRY_BATCH_SIZE]
                outputs = reward_pipe(batch_texts, **pipe_kwargs)

                if isinstance(outputs[0], dict):
                    batch_scores = [o["score"] for o in outputs]
                elif isinstance(outputs[0][0], dict):
                    batch_scores = [o[0]["score"] for o in outputs]
                else:
                    batch_scores = [float(o[0].cpu().numpy()) if hasattr(o[0], 'cpu') else float(o[0]) for o in outputs]
                scores.extend(batch_scores)

            del model, tokenizer, reward_pipe
            torch.cuda.empty_cache()
            gc.collect()

            all_raw_scores[(pm, rm)] = scores
            print(f"     SUCCESS: mean={np.mean(scores):.3f}, std={np.std(scores):.3f}")

        except Exception as e:
            import traceback
            print(f"     RETRY FAILED: {e}")
            print(traceback.format_exc())
            print("     Scores remain as zeros — consider reducing MAX_LENGTH or using CPU for this pair.")

    # Summary
    print(f"\n{'='*60}")
    print("Retry summary:")
    print(f"{'='*60}")
    for pm, rm in failed_pairs:
        scores = all_raw_scores[(pm, rm)]
        still_zero = all(s == 0.0 for s in scores)
        status = "STILL FAILED" if still_zero else "FIXED"
        print(f"  {pm:15s} + {rm:25s}: {status}  (mean={np.mean(scores):.3f})")
    print(f"\nRe-run Cell 5 (normalization) and Cell 6 (save) to update the output.")

In [ ]:
# Cell 5: Z-score normalization
# For each reward model, combine all 250 x 11 scores and normalize

available_prod_models = [m for m in MODEL_COLUMNS if m in df.columns]
reward_model_names = list(REWARD_MODELS.keys())

normalized_scores = {}  # key = (prod_model, reward_model)

for rm in reward_model_names:
    # Gather all scores for this reward model across all production models
    all_scores_for_rm = []
    for pm in available_prod_models:
        key = (pm, rm)
        if key in all_raw_scores:
            all_scores_for_rm.extend(all_raw_scores[key])

    combined = np.array(all_scores_for_rm, dtype=float)
    rm_mean = combined.mean()
    rm_std = combined.std()

    print(f"{rm:25s} - mean: {rm_mean:8.3f}, std: {rm_std:6.3f}, n={len(combined)}")

    for pm in available_prod_models:
        key = (pm, rm)
        if key in all_raw_scores:
            raw = np.array(all_raw_scores[key], dtype=float)
            if rm_std == 0:
                normalized_scores[key] = np.zeros_like(raw).tolist()
            else:
                normalized_scores[key] = ((raw - rm_mean) / rm_std).tolist()

# Compute per-production-model R_pref mean (average across reward models)
rpref_means = {}
for pm in available_prod_models:
    rm_scores = []
    for rm in reward_model_names:
        key = (pm, rm)
        if key in normalized_scores:
            rm_scores.append(normalized_scores[key])

    if rm_scores:
        stacked = np.array(rm_scores)  # shape: (n_reward_models, 250)
        rpref_means[pm] = stacked.mean(axis=0).tolist()
        print(f"  {pm}: R_pref mean = {np.mean(rpref_means[pm]):.4f}")

print(f"\nZ-score normalization complete for {len(available_prod_models)} production models")

In [ ]:
# Cell 6: Save cross_model_rpref.parquet

# Build output dataframe with prompt_id as join key
rpref_df = pd.DataFrame({"prompt_id": df["prompt_id"]})

# Add per-model R_pref mean columns
for pm in available_prod_models:
    if pm in rpref_means:
        rpref_df[f"rpref_mean_{pm}"] = rpref_means[pm]

# Add per-(production_model, reward_model) normalized scores for transparency
for pm in available_prod_models:
    for rm in reward_model_names:
        key = (pm, rm)
        if key in normalized_scores:
            rpref_df[f"rpref_{pm}_{rm}"] = normalized_scores[key]

rpref_df.to_parquet(OUTPUT_PATH, index=False)
print(f"Saved R_pref scores to: {OUTPUT_PATH}")
print(f"Shape: {rpref_df.shape}")
print(f"Columns: {list(rpref_df.columns[:15])}...")

# Summary
print(f"\n{'='*60}")
print("R_pref Summary by Production Model")
print(f"{'='*60}")
for pm in available_prod_models:
    col = f"rpref_mean_{pm}"
    if col in rpref_df.columns:
        vals = rpref_df[col].values
        print(f"  {pm:15s}: mean={np.mean(vals):+.4f}, std={np.std(vals):.4f}")